In [3]:
# Connexion à SQL Server
from sqlalchemy import create_engine
import pyodbc
# Libraries scientifiques (math)
import urllib
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
# Fichier utile (système et variable d'environnement)
from dotenv import load_dotenv
import sys
import os
from datetime import datetime
import csv


# Recharger variables d'envronnements
load_dotenv(override=True)
#1. Chemin du repertoire dans lequel se trouve le fichier
sys.path.append(os.getenv("DBCONNECT_PARENT"))  # The parent of `dbconnect`
#2. Module pour la connection à Sqlserver
from dbconnect.connection import SQLServerConnector
# 2.1 Récupération des paramètres de connection
conn = SQLServerConnector()
# print(conn.get_connection_string())
# Build connection string
conn_str = conn.get_connection_string()
params = urllib.parse.quote_plus(conn_str)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

sys.path.append('../dbconnect')
sys.path.append('../filepath')
# Connexion à SQL Server
conn = pyodbc.connect("DRIVER={ODBC Driver 18 for SQL Server};"
                      "SERVER=localhost,1433;"
                      "DATABASE=health;"
                      "UID=sa;"
                      "PWD=Mouscron2025?;"
                      "Encrypt=yes;"
                      "TrustServerCertificate=yes;")

cursor = conn.cursor()
# Compteurs
inserted_rows = 0
error_rows = 0
# Charger les données depuis un CSV
filecsv = os.environ.get("ETL_CSV_FILE")
fichier_csv= filecsv + "/" + 'patients_10000.csv'


# Fichier de log des erreurs
error_log_file = 'etl_errors.log'
with open(fichier_csv, newline='', encoding='utf-8')  as csvfile, \
     open(error_log_file, 'w', encoding='utf-8') as log_file:
    reader = csv.DictReader(csvfile)
    for row in reader:
        try:
            cursor.execute("""
                INSERT INTO dbo.Patients (
                    FirstName, LastName, DateOfBirth, Gender,
                    PhoneNumber, Email, Address,
                    EmergencyContactName, EmergencyContactPhone,
                    BloodType, Allergies, MedicalHistory, RegistrationDate
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, 
            row['FirstName'], row['LastName'], row['DateOfBirth'], row['Gender'],
            row['PhoneNumber'], row['Email'], row['Address'],
            row['EmergencyContactName'], row['EmergencyContactPhone'],
            row['BloodType'], row['Allergies'], row['MedicalHistory'], row['RegistrationDate'])

        except Exception as e:
            print(f"❌ Erreur sur la ligne {row}: {e}")
            print (e)

            # Transformer la ligne en DataFrame
            df = pd.DataFrame([row])  # On place le dict dans une liste

            # Sauvegarder ou appendre à un CSV d'erreurs
            df.to_csv('error_insert_patients.csv', mode='a', index=False, encoding='utf-8', header=not os.path.exists('error_insert_patients.csv'))

                
        continue  # Continue avec la ligne suivante

conn.commit()
cursor.close()
conn.close()
print("✅ ETL terminé.")


✅ ETL terminé.
